### **Day 9: Spark SQL & Relational Queries**

Yesterday, we mastered Window Functions and looked at how to compute complex analytics across row groups. Today, we are going to look at **Spark SQL**.

Up to this point, we have discussed PySpark as a programmatic API where you manipulate DataFrames using Python methods. However, one of Spark’s greatest strengths is its unified nature. Spark includes a native, fully ANSI-compliant SQL query engine. This means you can register any distributed DataFrame as a temporary relational view and query it using standard SQL syntax.

Whether you write programmatic Python code or relational SQL strings, Spark executes them using the exact same underlying architecture.

**Today's Objective**

By the end of this session, you will understand how the Spark SQL engine bridges programmatic code with relational queries, how to create temporary and global views, and how Spark compiles SQL strings into optimized distributed tasks.

**1. The Unified Engine: SQL vs. DataFrames**

A common misconception is that Spark SQL is slower or fundamentally different from the programmatic DataFrame API. In reality, they are two different interfaces for the exact same engine.

When you write a query in Spark SQL, like `SELECT department, AVG(salary) FROM employees GROUP BY department`, a component called the **Catalyst Optimizer** parses the text string. It translates that string into the exact same logical execution plan (DAG) that would be generated if you had written `df.groupBy("department").avg("salary")`.

Because both approaches compile down to the identical underlying Java bytecode executed by the worker nodes, there is **zero performance difference** between writing pure PySpark code and writing pure SQL queries. You can choose whichever style fits your pipeline design or team skillset.

**2. Registering Temporary Views**

To run SQL queries against a DataFrame, you must first create a relational entry point for the SQL engine. You cannot just run a query directly against a Python variable name. You must register the DataFrame as a **Temporary View**.

A Temporary View acts as an alias or a pointer to your DataFrame. It lives inside the catalog of your specific `SparkSession`.

*Local Temporary Views*

* **How it works:** You use the method `df.createOrReplaceTempView("my_sql_table")`.
* **Scope:** This view is tied directly to the specific `SparkSession` that created it. If your script spins up a second Spark session, or if you are running a multi-tenant cluster where another user has their own session, they cannot see or query your local temporary view. It disappears completely when your application stops.

*Global Temporary Views*

* **How it works:** You use the method `df.createOrReplaceGlobalTempView("my_global_table")`.
* **Scope:** This view is shared across *all* Spark sessions running within the same Spark application cluster. Spark stores global views inside a system-reserved database called `global_temp`. To query it, you must explicitly reference the system prefix: `SELECT * FROM global_temp.my_global_table`.

**3. Mixing Python Code with SQL**

Because Spark SQL and DataFrames share the same engine, they can be blended seamlessly within the same script. The output of a Spark SQL query is **always a DataFrame**. This allows you to perform initial data cleaning using Python, switch to SQL for complex aggregations, and return to Python to save the file.

Let's walk through the conceptual architecture of a blended pipeline:

```python
# 1. Load data using the Python DataFrame API
df = spark.read.parquet("raw_sales_data.parquet")

# 2. Register the DataFrame so the SQL engine can see it
df.createOrReplaceTempView("sales_records")

# 3. Execute a standard SQL query string. 
# This returns a brand new distributed DataFrame object.
sql_results_df = spark.sql("""
    SELECT Region, SUM(Revenue) as Total_Revenue 
    FROM sales_records 
    WHERE Status = 'Completed' 
    GROUP BY Region 
    ORDER BY Total_Revenue DESC
""")

# 4. Continue manipulating the result using Python methods
final_df = sql_results_df.filter(sql_results_df["Total_Revenue"] > 100000)

# 5. Save the output
final_df.write.mode("overwrite").csv("processed_revenue_report/")

```